<a href="https://colab.research.google.com/github/santhoshml/analyse-human-traits-from-tweet/blob/main/Analyse_past_tweets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip install git+https://github.com/tweepy/tweepy.git --upgrade
!pip install openai langsmith -q

  Cloning https://github.com/tweepy/tweepy.git to /tmp/pip-req-build-8g12jcz9
  Running command git clone --filter=blob:none --quiet https://github.com/tweepy/tweepy.git /tmp/pip-req-build-8g12jcz9
  Resolved https://github.com/tweepy/tweepy.git to commit db28c0e84826485755eb7fcef0c30f75395dff5f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [15]:
# Import the tweepy package
import tweepy

# Import a variety of other packages that may be useful for working with data.
import pandas as pd
import json
import time
import os
import getpass
from google.colab import userdata
from openai import OpenAI
import re
from collections import Counter
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from uuid import uuid4


In [16]:
my_consumer_key = userdata.get('X_CONSUMER_KEY')
my_consumer_secret = userdata.get('X_CONSUMER_SECRET')

my_access_token = userdata.get('X_ACCESS_TOKEN')
my_access_secret = userdata.get('X_ACCESS_SECRET')

my_bearer_token = userdata.get('X_BEARER_TOKEN')

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

unique_id = uuid4().hex[0:8]
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = f"Analyse-tweets-{unique_id}"
os.environ["LANGCHAIN_API_KEY"] = userdata.get('LANGSMITH_API_KEY')

In [21]:
tweepyClient = tweepy.Client(
    wait_on_rate_limit = True,
    consumer_key = my_consumer_key,
    consumer_secret = my_consumer_secret,
    access_token = my_access_token,
    access_token_secret = my_access_secret,
    bearer_token = my_bearer_token,
)

openai_client = OpenAI()

In [ ]:
user_single = tweepyClient.get_user(
    username ="elonmusk",
)
user_single.data.id

In [ ]:
# Method 1 to retrieve past tweets using twitter api
import requests

headers = {
    "Authorization": f"Bearer {my_bearer_token}",
}

params = {
    "query": "from:elonmusk",  # change username or query
    "max_results": 100,
    "tweet.fields": "created_at,text",
    "start_time": "2021-01-01T00:00:00Z",  # Optional: ISO 8601 format
    "end_time": "2024-12-31T00:00:00Z",    # Optional
}

url = "https://api.twitter.com/2/tweets/search/all"

response = requests.get(url, headers=headers, params=params)
data = response.json()
print(data)

for tweet in data.get("data", []):
    print(f"{tweet['created_at']}: {tweet['text']}")


In [ ]:
# Method 2 to retrieve past 100 tweets for a user based on user_id.
# This uses tweepy

def get_all_tweets(user_id, tweet_fields):
    """
    Fetches all tweets for a given user, handling rate limits.
    """
    all_tweets = []
    next_token = None
    while True:
        try:
            # Include `max_results` for pagination (max 100 per request)
            response = tweepyClient.get_users_tweets(
                id=user_id,
                tweet_fields=tweet_fields,
                max_results=100, # Get 100 tweets per request (max allowed)
                pagination_token=next_token,
            )
            if response.data:
                all_tweets.extend(response.data)
            else:
                # No more tweets, break the loop
                break
            # Get the next token for pagination
            next_token = response.meta.get('next_token')
            if not next_token:
                # No next token, we've retrieved all tweets
                break

        except tweepy.TooManyRequests as e:
            print("Rate limit exceeded. Waiting for 15 minutes...")
            time.sleep(900)  # Wait for 15 minutes before retrying
            continue  # Continue the loop after waiting

    return all_tweets

single_user_tweets = get_all_tweets(
    user_id = user_single.data.id ,
    tweet_fields = ["id","created_at", "text","public_metrics",
                    "in_reply_to_user_id","reply_settings", "source",
                    "referenced_tweets"])

single_user_tweets_df = pd.DataFrame()

for i in single_user_tweets.data:
  temp_df = pd.json_normalize(i.data, sep="_")
  # Use pd.concat instead of append
  single_user_tweets_df = pd.concat([single_user_tweets_df, temp_df], ignore_index=True)

separator = ', '
concatenated_string = separator.join(single_user_tweets_df['text'].astype(str))

In [43]:
# Read elonmuck tweet's form csv
# tweets = pd.read_csv('elon_musk_tweets.csv')
# tweets = pd.read_csv('Tweets-BarackObama.csv')
tweets = pd.read_csv('jimmyfallon.csv')

tweets.head()

def remove_mentions(text: str):
  return re.sub(r'@\w+', '', text)

def remove_links(text: str):
  return re.sub(r'https?://\S+|www\.\S+', '', text)

def remove_amp(text: str):
  return re.sub(r'&amp;', '', text)

def cleanup_strings(text: str):
  return remove_amp(remove_links(remove_mentions(text)))

def get_word_count(line):
  return len(line.split())

def cosine_similarity(text1, text2):
    vectorizer = CountVectorizer()
    # Convert text1 and text2 to lowercase before fitting
    vectorizer.fit([text1.lower(), text2.lower()])
    vector1 = vectorizer.transform([text1]).toarray()
    vector2 = vectorizer.transform([text2]).toarray()

    # Use imported cosine_similarity function
    from sklearn.metrics.pairwise import cosine_similarity
    similarity_score = cosine_similarity(vector1, vector2)[0][0]
    return similarity_score

def is_very_similar(text, arr):
  max_similarity = 0.2 # max allowed similarity between any 2 tweets is 0.2
  for i in arr:
    similarity = cosine_similarity(text, i)
    if similarity > max_similarity:
      return True
  return False

def get_nonsimilar_recent_tweets():
  count = 0  # Initialize a counter variable
  concatenated_string = ''  # Initialize an empty string
  arr=[]
  for index, row in tweets.iterrows(): # Iterate through rows using iterrows()
      if count < 100:  # Check if the counter is less than 100
        text = cleanup_strings(row['text'])
        # print(f"text:{text}", text)
        is_similar = is_very_similar(text, arr)
        if get_word_count(text) > 9 and not is_similar:
          arr.append(text)
          count += 1  # Increment the counter
      else:
          break  # Exit the loop if 100 tweets have been processed
  concatenated_string = ". ".join(arr)
  # print(concatenated_string)
  return concatenated_string

def get_all_with_n_retweets(n):
  count = 0  # Initialize a counter variable
  concatenated_string = ''  # Initialize an empty string
  arr=[]
  for index, row in tweets.iterrows(): # Iterate through rows using iterrows()
      if count < 100:  # Check if the counter is less than 100
        text = cleanup_strings(row['text'])
        if row['retweets'] > n and get_word_count(text) > 9:
          arr.append(text)
          count += 1  # Increment the counter
      else:
          break  # Exit the loop if 100 tweets have been processed
  concatenated_string = ". ".join(arr)
  return concatenated_string

def get_recent_tweets():
  count = 0  # Initialize a counter variable
  concatenated_string = ''  # Initialize an empty string
  arr=[]
  for index, row in tweets.iterrows(): # Iterate through rows using iterrows()
      if count < 100:  # Check if the counter is less than 100
        text = cleanup_strings(row['text'])
        if get_word_count(text) > 9:
          arr.append(text)
          count += 1  # Increment the counter
      else:
          break  # Exit the loop if 100 tweets have been processed
  concatenated_string = ". ".join(arr)
  return concatenated_string

def get_simply_recent_tweets():
  count = 0  # Initialize a counter variable
  concatenated_string = ''  # Initialize an empty string
  for index, row in tweets.iterrows(): # Iterate through rows using iterrows()
      if count < 100:  # Check if the counter is less than 100
        concatenated_string += row['text']
        count += 1  # Increment the counter
      else:
          break  # Exit the loop if 100 tweets have been processed
  return concatenated_string



In [44]:
def system_prompt(message: str) -> dict:
    return {"role": "system", "content": message}

def user_prompt(message: str) -> dict:
    return {"role": "user", "content": message}


def get_response_from_gpt(concatenated_string: str) -> str:
  evaluator_system_template = """You are an expert in analyzing the basic human traits based on the past tweets.

  You should be hyper-critical.

  Provide scores (out of 10) for the following attributes:

  Here's a more detailed look at some key basic human traits:
  Creativity: The ability to use imagination and unique ideas to create something new.
  Self-awareness: Understanding one's own thoughts, feelings, and motivations.
  Adaptability: The capacity to adjust to changing circumstances and environments.
  Empathy: The ability to understand and share the feelings of others.
  Communication: The ability to express oneself and understand others through language and other forms of expression.
  Learning: The capacity to acquire knowledge and skills.
  Sociality: Humans are inherently social beings, needing meaningful relationships and community.
  Intelligence: The ability to acquire and apply knowledge and skills.
  Courage: The ability to stand up for what one believes in, overcome obstacles, and take risks.
  Integrity: The quality of being honest and having strong moral principles.
  Optimism: A tendency to look at the positive side of things and believe in the possibility of good outcomes.
  Resilience: The ability to bounce back from adversity and challenges.
  Passion: A strong feeling of enthusiasm or excitement for something.


  Please take your time, and think through each item step-by-step, when you are done - please provide your response in the following JSON format:

  {"Creativity" : "score_out_of_10", "Self-awareness" : "score_out_of_10", "Adaptability" : "score_out_of_10",
  "Empathy" : "score_out_of_10", "Communication" : "score_out_of_10", "Learning" : "score_out_of_10",
  "Sociality" : "score_out_of_10", "Intelligence" : "score_out_of_10", "Courage" : "score_out_of_10",
  "Integrity" : "score_out_of_10", "Optimism" : "score_out_of_10", "Resilience" : "score_out_of_10", "Passion" : "score_out_of_10"}"""

  evaluation_template = """Query: {input}"""

  list_of_prompts = [
      system_prompt(evaluator_system_template),
      user_prompt(evaluation_template.format(input=concatenated_string))
  ]

  evaluator_response = openai_client.chat.completions.create(
      model="gpt-4o",
      temperature=0,
      messages=list_of_prompts,
      response_format={"type" : "json_object"}
  )
  return evaluator_response

In [45]:
# Get the JSON content from the response
json_content_simply_recent_tweets = get_response_from_gpt(get_simply_recent_tweets()).choices[0].message.content
json_content_recent_tweets = get_response_from_gpt(get_recent_tweets()).choices[0].message.content
json_content_recent_nonsimilar_tweets = get_response_from_gpt(get_nonsimilar_recent_tweets()).choices[0].message.content
json_content_with_n_retweets = get_response_from_gpt(get_all_with_n_retweets(100)).choices[0].message.content


data_simply_recent_tweets = json.loads(json_content_simply_recent_tweets)
data_recent_tweets = json.loads(json_content_recent_tweets)
data_recent_nonsimilar_tweets = json.loads(json_content_recent_nonsimilar_tweets)
data_with_n_retweets = json.loads(json_content_with_n_retweets)


# 2. Create DataFrames for each JSON object:
df_simply_recent_tweets = pd.DataFrame([data_simply_recent_tweets]).T
df_recent_tweets = pd.DataFrame([data_recent_tweets]).T
df_recent_nonsimilar_tweets = pd.DataFrame([data_recent_nonsimilar_tweets]).T
df_with_n_retweets = pd.DataFrame([data_with_n_retweets]).T

# 3. Rename columns:
df_simply_recent_tweets.columns = ['Scores_Simply_Recent_Tweets']
df_recent_tweets.columns = ['Scores_Recent_Tweets']
df_recent_nonsimilar_tweets.columns = ['Scores_Recent_nonsimilar_Tweets']
df_with_n_retweets.columns = ['Scores_With_N_Retweets']


# 4. Merge the DataFrames on index (keys):
merged_df1 = pd.merge(df_simply_recent_tweets, df_recent_tweets, left_index=True, right_index=True, how='outer')
merged_df2 = pd.merge(merged_df1, df_recent_nonsimilar_tweets, left_index=True, right_index=True, how='outer')
merged_df = pd.merge(merged_df2, df_with_n_retweets, left_index=True, right_index=True, how='outer')

# 5. Display the merged DataFrame:
display(merged_df)

,Scores_Simply_Recent_Tweets,Scores_Recent_Tweets,Scores_Recent_nonsimilar_Tweets,Scores_With_N_Retweets
Adaptability,7,7,7,7
Communication,9,9,9,9
Courage,6,6,5,6
Creativity,8,8,8,8
Empathy,6,7,5,7
Integrity,7,7,6,7
Intelligence,7,7,7,7
Learning,6,6,6,6
Optimism,8,8,8,8
Passion,8,8,7,8
